In [36]:
"""
profile_raw.py -- profile the RAW (pre-cleaning) data for the report.

Usage:
    python src/profile_raw.py [raw_json] [out_dir]
Writes profiling tables as CSV into out_dir and prints a summary.
"""
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

RAW = Path(sys.argv[1] if len(sys.argv) > 1 else "data/raw/raw_countries_.json")
OUT = Path(sys.argv[2] if len(sys.argv) > 2 else "data/profile")
OUT.mkdir(parents=True, exist_ok=True)

with open('raw_countries_.json', encoding="utf-8") as f:
    recs = json.load(f)

if recs and "names" not in recs[0]:
    if "names_common" not in recs[0]:
        sys.exit("ERROR: unrecognised input format (expected nested API JSON or the flattened table).")
    print("WARNING: input is the FLATTENED file; profiling it as-is.")

    def _split(v):
        return [x for x in (v or "").split(", ") if x]

    recs = [{
        "names": {"common": r.get("names_common")},
        "codes": {"alpha_3": r.get("codes_alpha_3")},
        "region": r.get("region"),
        "population": r.get("population"),
        "area": {"kilometers": r.get("area_kilometers")},
        "borders": _split(r.get("borders")),
        "languages": [{"bcp47": c} for c in _split(r.get("languages"))],
        "currencies": [{"name": n} for n in _split(r.get("currencies"))],
    } for r in recs]

df = pd.DataFrame([{
    "name": (r.get("names") or {}).get("common"),
    "Codes": (r.get("codes") or {}).get("alpha_3"),
    "region": r.get("region"),
    "population": r.get("population"),
    "area_km2": (r.get("area") or {}).get("kilometers"),
    "n_borders": len(r.get("borders") or []),
    "n_languages": len(r.get("languages") or []),
    "n_currencies": len(r.get("currencies") or []),
} for r in recs])

print(f"Shape: {df.shape}")

# 1. data types
dtypes = df.dtypes.astype(str).rename("dtype").to_frame()
dtypes.to_csv(OUT / "dtypes.csv")
print("\n== dtypes ==\n", dtypes)

# 2. missingness (treat '' and None as missing; 0 counts for list-derived columns)
miss = pd.DataFrame({
    "missing_or_empty": [
        df["name"].isna().sum() + (df["name"] == "").sum(),
        df["Codes"].isna().sum() + (df["Codes"] == "").sum(),
        df["region"].isna().sum() + (df["region"] == "").sum(),
        df["population"].isna().sum(),
        df["area_km2"].isna().sum(),
        (df["n_borders"] == 0).sum(),
        (df["n_languages"] == 0).sum(),
        (df["n_currencies"] == 0).sum(),
    ]
}, index=["name", "Codes", "region", "population", "area_km2",
          "borders (0 = no land border)", "languages", "currencies"])
miss["pct"] = (miss["missing_or_empty"] / len(df) * 100).round(1)
miss.to_csv(OUT / "missingness.csv")
print("\n== missingness ==\n", miss)

print("\nRows with empty Codes:", df.loc[df["Codes"].isin(["", None]), "name"].tolist())
print("Rows with no currency:", df.loc[df["n_currencies"] == 0, "name"].tolist())
print("Rows with no language:", df.loc[df["n_languages"] == 0, "name"].tolist())
print("Population == 0:", df.loc[df["population"] == 0, "name"].tolist())

# 3. summary statistics
desc = df[["population", "area_km2", "n_borders", "n_languages", "n_currencies"]].describe().T
desc["skew"] = df[["population", "area_km2", "n_borders", "n_languages", "n_currencies"]].skew()
desc.to_csv(OUT / "summary_stats.csv")
print("\n== summary stats ==\n", desc.round(2).to_string())

# 4. outliers (IQR on log10 scale + top/bottom lists)
rows = []
for col in ["population", "area_km2"]:
    s = df.loc[df[col] > 0, col]
    lg = np.log10(s)
    q1, q3 = lg.quantile([.25, .75])
    lo, hi = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
    out = df.loc[s.index][(lg < lo) | (lg > hi)]
    rows.append({"column": col, "fence_low": round(10 ** lo), "fence_high": round(10 ** hi),
                 "n_outliers": len(out), "outliers": "; ".join(out["name"])})
    print(f"\nTop 5 {col}:", df.nlargest(5, col)["name"].tolist())
    print(f"Bottom 5 {col} (>0):", df[df[col] > 0].nsmallest(5, col)["name"].tolist())
pd.DataFrame(rows).to_csv(OUT / "outliers.csv", index=False)

# 5. region counts
reg = df["region"].value_counts().rename_axis("region").reset_index(name="count")
reg.to_csv(OUT / "region_counts.csv", index=False)
print("\n== region counts ==\n", reg)

# 6. integrity checks
print("\nDuplicate names:", df["name"].duplicated().sum())
codes = set(df["Codes"].replace("", np.nan).dropna())
bad = {b for r in recs for b in (r.get("borders") or []) if b not in codes}
print("Border codes not in dataset:", bad or "none")
display(df)

Shape: (254, 8)

== dtypes ==
                 dtype
name              str
Codes             str
region            str
population      int64
area_km2      float64
n_borders       int64
n_languages     int64
n_currencies    int64

== missingness ==
                               missing_or_empty   pct
name                                         0   0.0
Codes                                        4   1.6
region                                       0   0.0
population                                   0   0.0
area_km2                                     0   0.0
borders (0 = no land border)                86  33.9
languages                                    1   0.4
currencies                                   3   1.2

Rows with empty Codes: ['Abkhazia', 'Northern Cyprus', 'Somaliland', 'South Ossetia']
Rows with no currency: ['Antarctica', 'Bouvet Island', 'Heard Island and McDonald Islands']
Rows with no language: ['Antarctica']
Population == 0: ['Bouvet Island', 'Heard Island and McDo

,name,Codes,region,population,area_km2,n_borders,n_languages,n_currencies
0,Abkhazia,,Asia,244236,8665.0,2,2,1
1,Afghanistan,AFG,Asia,43844000,652230.0,6,3,1
2,Albania,ALB,Europe,2335930,28748.0,4,1,1
3,Algeria,DZA,Africa,47400000,2381741.0,7,1,1
4,American Samoa,ASM,Oceania,43268,199.0,0,2,1
...,...,...,...,...,...,...,...,...
249,Western Sahara,ESH,Africa,510713,266000.0,3,2,3
250,Yemen,YEM,Asia,32684503,527968.0,2,1,1
251,Zambia,ZMB,Africa,23003214,752612.0,8,1,1
252,Zimbabwe,ZWE,Africa,16175966,390757.0,4,15,2


In [37]:
j=df['name']=='Jordan'
df[j]

,name,Codes,region,population,area_km2,n_borders,n_languages,n_currencies
114,Jordan,JOR,Asia,11937000,89342.0,5,1,1
